# 21장 딥러닝을 이용한 자연어 처리

1.1) 자연어 처리 (Natural Language Processing, NLP) = 컴퓨터 인공지능이, 인간의 문장을 듣고, 무엇을 의미하는 지 이해하는 것

-자연어 우리가 평소에 말하는 음성이나 텍스트

---

2.1.)자연어 처리의 첫 단계

= 텍스트의 토큰화 (텍스트를 잘게 나누는 것)

= 입력할 텍스트를, 단어별/문장별/형태소별로 나누는 것. 이 각각의 단위를 **토큰**이라고 한다.

=입력된 텍스트를 잘게 나누는 과정 = **토큰화**

=케라스의 **text_to_word_sequence()** 함수 사용

## 1. 텍스트의 토큰화

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Flatten,Embedding
from tensorflow.keras.utils import to_categorical
from numpy import array

# 케라스의 텍스트 전처리와 관련한 함수 중 text_to_word_sequence 함수를 불러옵니다.
from tensorflow.keras.preprocessing.text import text_to_word_sequence

# 전처리할 텍스트를 정합니다.
text = '해보지 않으면 해낼 수 없다'

# 해당 텍스트를 토큰화합니다.
result = text_to_word_sequence(text)
print("\n원문:\n", text)
print("\n토큰화:\n", result)


원문:
 해보지 않으면 해낼 수 없다

토큰화:
 ['해보지', '않으면', '해낼', '수', '없다']



---
1.1.)이렇게 주어진 텍스트를 단어 단위로 쪼개고 나면, 각 단아거 몇 번이나 중복해서 쓰였는지 그 **빈도수**를 알 수 있다.

= **해당 텍스트에서 중요한 역할을 하는 단어를 파악할 수 있는 것**


1.2.) 방법 = **Bag-of-Words** = 같은 단어 끼리 따로 따로 가방에 담은 후, 각 가방에 몇 개의 단어가 들어있는지 세는 기법 **Tokenizer()**

In [ ]:
# 단어 빈도수 세기

# 전처리하려는 세 개의 문장을 정합니다.
docs = ['먼저 텍스트의 각 단어를 나누어 토큰화합니다.',
       '텍스트의 단어로 토큰화해야 딥러닝에서 인식됩니다.',
       '토큰화한 결과는 딥러닝에서 사용할 수 있습니다.',
       ]

# 토큰화 함수를 이용해 전처리 하는 과정입니다.
token = Tokenizer()            # 토큰화 함수 지정
token.fit_on_texts(docs)       # 토큰화 함수에 문장 적용

# 단어의 빈도수를 계산한 결과를 각 옵션에 맞추어 출력합니다.
# Tokenizer()의 word_counts 함수는 순서를 기억하는 OrderedDict 클래스를 사용합니다.
print("\n단어 카운트:\n", token.word_counts)
##결과적으로 '토큰화'라는 단어가 3회, '텍스트의', '딥러닝에서'가 2회, 나머지가 1회씩 나오고 있음.

# 출력되는 순서는 랜덤입니다.
print("\n문장 카운트: ", token.document_count) #몇 문장인지 카운트
print("\n각 단어가 몇 개의 문장에 포함되어 있는가:\n", token.word_docs) #각 단어가 몇 개의 문장에 포함되는가
print("\n각 단어에 매겨진 인덱스 값:\n",  token.word_index) #각 단어에 매겨진 인덱스 값이란 무엇인가?? (이는 단어 임베딩 부분에서 추가적으로 사용할 것)


단어 카운트:
 OrderedDict([('먼저', 1), ('텍스트의', 2), ('각', 1), ('단어를', 1), ('나누어', 1), ('토큰화합니다', 1), ('단어로', 1), ('토큰화해야', 1), ('딥러닝에서', 2), ('인식됩니다', 1), ('토큰화한', 1), ('결과는', 1), ('사용할', 1), ('수', 1), ('있습니다', 1)])

문장 카운트:  3

각 단어가 몇 개의 문장에 포함되어 있는가:
 defaultdict(<class 'int'>, {'토큰화합니다': 1, '먼저': 1, '나누어': 1, '텍스트의': 2, '각': 1, '단어를': 1, '토큰화해야': 1, '인식됩니다': 1, '딥러닝에서': 2, '단어로': 1, '있습니다': 1, '사용할': 1, '토큰화한': 1, '수': 1, '결과는': 1})

각 단어에 매겨진 인덱스 값:
 {'텍스트의': 1, '딥러닝에서': 2, '먼저': 3, '각': 4, '단어를': 5, '나누어': 6, '토큰화합니다': 7, '단어로': 8, '토큰화해야': 9, '인식됩니다': 10, '토큰화한': 11, '결과는': 12, '사용할': 13, '수': 14, '있습니다': 15}


## 2. 단어의 원-핫 인코딩
**역할**: 컴퓨터가 이해할 수 있는 가장 작은 단위의 벡터를 만드는 것입니다.

1.1.) 앞서 배운 원-핫 인코딩은, 다중 분류 문제에서와 같이 결과 값이 3개 이상일 때, 그 것을 0-1의 배열로 표현하는 것이다. [100][010][001]

1.2.) 텍스트에서 **원-핫 인코딩이란 각 단어를 모두 0으로 바꾸어주고, 원하는 단어만 1로 바꾸어주는 것.**

1.3.) 방법: 먼저 단어 수만큼 0으로 채워진 벡터 공간을 바꿀 것 -> 각 단어가 벡터 배열에 해당하는 위치를 1로 바꾸면 됨.

In [ ]:
text="오랫동안 꿈꾸는 이는 그 꿈을 닮아간다"
token = Tokenizer()
token.fit_on_texts([text])
print(token.word_index)

{'오랫동안': 1, '꿈꾸는': 2, '이는': 3, '그': 4, '꿈을': 5, '닮아간다': 6}


In [ ]:
x=token.texts_to_sequences([text]) #토큰의 인덱스로만 채워진 새로운 배열을 만드는 것
print(x)

[[1, 2, 3, 4, 5, 6]]


In [ ]:
# 인덱스 수에 하나를 추가해서 원-핫 인코딩 배열 만들기
word_size = len(token.word_index) + 1 #배열 맨 앞에 0이 추가되므로, 단어 수보다 1이 더 많게 인덱스 숫자를 잡아주어야 함
x = to_categorical(x, num_classes=word_size)
print(x)

[[[0. 1. 0. 0. 0. 0. 0.]
  [0. 0. 1. 0. 0. 0. 0.]
  [0. 0. 0. 1. 0. 0. 0.]
  [0. 0. 0. 0. 1. 0. 0.]
  [0. 0. 0. 0. 0. 1. 0.]
  [0. 0. 0. 0. 0. 0. 1.]]]


##단어 임베딩

**역할**: 컴퓨터가 이해할 수 있는 가장 작은 단위의 벡터를 만드는 것입니다.

1.1.) 원-핫 인코딩을 그대로 사용하면, 벡터의 길이가 너무 길어짐.. 1만 개의 단어 토큰이라면, 0을 9,999번 쓰고 하나의 1로만 이루어진 단어 벡터를 1만 개나 만들어야 하는 것.

1.2.)이러한 공간적 낭비를 해결하기 위해, **단어 임베딩**이 나옴.
= **각 단어의 유사도를 바탕으로, 주어진 배열을 정해진 길이로 압축시키는 것 Embedding() 함수**

1.3.)단어 간 유사도는 학습과 오차 역전파를 통해 계산할 수 있다. = e.g., 적절한 크기의 배열을 바꾸어 주기 위해 최적의 유사도를 계산하는 학습과정을 거치는 것.



💡 단어 임베딩(Word Embedding): 도대체 무엇을 학습하는가?

원-핫 인코딩의 한계를 넘어, 단어를 **밀집 벡터(Dense Vector)**로 변환하는 과정에서 모델은 무엇을 배우는 것일까요?

---
1. 학습 대상: 임베딩 층(Embedding Layer)의 '좌표'

딥러닝 모델 안에는 단어의 좌표를 담은 거대한 **표(임베딩 층)**가 있습니다.

초기 상태: 모든 단어의 좌표가 **랜덤(Random)**하게 찍혀 있습니다. (예: '사과'와 '비행기'가 우연히 비슷할 수 있음)

학습 목표: "비슷한 의미를 가진 단어는 비슷한 좌표(숫자)를 갖도록 표를 수정하라!"

---
2. 학습 과정: 오차 역전파를 통한 '의미 이동'

문맥을 통해 단어의 유사도를 계산하고 좌표를 수정합니다.

예시 상황: "나는 맛있는 [ ? ] 를 먹었다."

예측 (Forward): 초기엔 멍청해서 **"비행기를 먹었다"**라고 예측할 수 있습니다.

오차 발견 (Loss): "틀렸어! 정답은 사과야. (비행기는 먹을 수 없어!)"

수정 (Backward): 역전파 알고리즘이 임베딩 층으로 되돌아와 **가중치(좌표)**를 수정합니다.

📉 비행기: "먹었다와 안 어울려. 좌표 저 멀리로 이동해."

📈 사과/바나나: "먹었다, 맛있는과 잘 어울려. 너희끼리 좌표를 가깝게 뭉쳐."

---

3. 최종 결과: '의미 공간(Semantic Space)'의 형성

이 과정을 수만 번 반복하면, 랜덤했던 숫자 표가 의미 있는 지도로 변합니다.

🍎 과일 구역: 사과, 배, 포도... (거리가 가까움)

✈️ 탈것 구역: 비행기, 자동차, 자전거... (과일 구역과 멂)

👑 왕족 구역: 왕, 여왕, 왕자...

In [ ]:
model = Sequential()
model.add(Embedding(16, 4)) #입력이 될 총 단어수 16, 임베딩 후 출력되는 벡터 크기는 4
model.build((None, 2)) #None은 한 번에 처리할 데이터 묶음의 크기임. None으로 설정해두면 나중에 필요에 따라 자유롭게 바꿀 수 있음
# 2 = 한 번에 처리할 단어의 개수 (단어 2개를 하나의 입력단위로 모델이 처리)

## 4.텍스트를 읽고 긍정, 부정 예측하기

In [ ]:
# 텍스트 리뷰 자료를 지정합니다.
docs = ["너무 재밌네요","최고예요","참 잘 만든 영화예요","추천하고 싶은 영화입니다","한번 더 보고싶네요","글쎄요","별로예요","생각보다 지루하네요","연기가 어색해요","재미없어요"]

# 긍정 리뷰는 1, 부정 리뷰는 0으로 클래스를 지정합니다.
classes = array([1,1,1,1,1,0,0,0,0,0])

# 토큰화
token = Tokenizer()
token.fit_on_texts(docs)
print(token.word_index)


{'너무': 1, '재밌네요': 2, '최고예요': 3, '참': 4, '잘': 5, '만든': 6, '영화예요': 7, '추천하고': 8, '싶은': 9, '영화입니다': 10, '한번': 11, '더': 12, '보고싶네요': 13, '글쎄요': 14, '별로예요': 15, '생각보다': 16, '지루하네요': 17, '연기가': 18, '어색해요': 19, '재미없어요': 20}


In [ ]:
x = token.texts_to_sequences(docs) #토큰에 지정된 인덱스로 새로운 배열을 만듦.
print("\n리뷰 텍스트, 토큰화 결과:\n",  x) #알다시피, 각 토큰별 길이는 단어의 개수에 따라 달라지는 것을 알 수 있음


리뷰 텍스트, 토큰화 결과:
 [[1, 2], [3], [4, 5, 6, 7], [8, 9, 10], [11, 12, 13], [14], [15], [16, 17], [18, 19], [20]]


In [ ]:
# 패딩, 서로 다른 길이의 데이터를 4로 맞추어 줍니다. = pad_sequence()함수
padded_x = pad_sequences(x, 4) #원하는 길이보다 짧은 부분은 0을 채우고, 긴 데이터는 잘라서 같은 길이로 맞춰주는 함수, #이 경우, 앞서 생성한 x 배열을 4개의 길이로 맞추겠다는 뜻
print("\n패딩 결과:\n", padded_x)


패딩 결과:
 [[ 0  0  1  2]
 [ 0  0  0  3]
 [ 4  5  6  7]
 [ 0  8  9 10]
 [ 0 11 12 13]
 [ 0  0  0 14]
 [ 0  0  0 15]
 [ 0  0 16 17]
 [ 0  0 18 19]
 [ 0  0  0 20]]


In [ ]:
#임베딩 함수에게는 3가지 파라미터가 필요하다 = 입력 (총 몇 개의 단어 집합에서), 출력 (몇 개의 인덱스 결과를 사용할 것인지), 단어수 (매번 입력될 단어 수는 몇 개로 할 것인지)
# 임베딩에 입력될 단어의 수를 지정합니다.
word_size = len(token.word_index) +1 #원-핫 인코딩 벡터를 만들기 위해서 전체 길이 +1이 필요하였음.
#위의 변수는 (1) 총 몇 개의 인덱스가 '입력'되어야 하는지 지정해주는 것

# 단어 임베딩을 포함하여 딥러닝 모델을 만들고 결과를 출력합니다.
model = Sequential()
model.add(Embedding(word_size, 8)) #임베딩 이후 출력될 크기는 랜덤으로 8로 맞춤
model.build((None, 4)) #한 번에 4개의 단어를 입력으로 받을 것임 (앞서 패딩에서 길이를 4로 맞춰주었음)
model.add(Flatten()) #2차원 행렬 형태의 데이터 (표)를 1차원 벡터 (한 줄)로 쭉 펴서 마지막 출력층에 전달하기 위함임.
model.add(Dense(1, activation='sigmoid'))
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 4, 8)           │           168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 201 (804.00 B)

 Trainable params: 201 (804.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(padded_x, classes, epochs=20) #20번 진행 후 결과 출력
print("\n Accuracy: %.4f" % (model.evaluate(padded_x, classes)[1]))

Epoch 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 495ms/step - accuracy: 0.5000 - loss: 0.6890
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.7000 - loss: 0.6869
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.7000 - loss: 0.6847
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.9000 - loss: 0.6825
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.9000 - loss: 0.6803
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.9000 - loss: 0.6781
Epoch 7/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.9000 - loss: 0.6760
Epoch 8/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.9000 - loss: 0.6738
Epoch 9/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.9000 - loss: 0.6716
Epoch 10/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.9000 - loss: 0.6693
Epoch 11/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.9000 - loss: 0.6671
Epoch 12/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.9000 - loss: 0.6649
